# Bangkok CCTV Vehicle Detection with YOLO11s (imgsz 960)
2110531 DSDE 2026/1 — Kaggle midterm

Follows the flow of the Week 05 notebook **3-2 Object detection with YOLOv8 (custom dataset)**
(`3_3_Object_detection_VOCDetection_yolov8_advanced.ipynb`):
**convert our dataset to YOLO format → write `data.yaml` → `model.train()` → check results → predict.**
Changes from the class notebook:
- `yolo11s.pt` instead of `yolov8n.pt` (same Ultralytics API, newer and bigger backbone)
- `imgsz=960`: images are only 352×288 and vehicles are tiny (median car 14×19 px), so we upscale ~3×
- validation split **by camera**, because the test set uses 5 cameras not seen in train
- the last section writes `submission.csv` in the Kaggle format

Runtime: Colab **T4 GPU** (Runtime → Change runtime type → T4 GPU).

# Setup

In [ ]:
%pip install -q ultralytics pycocotools
import ultralytics
ultralytics.checks()

In [ ]:
! nvidia-smi

## Get the data (no Kaggle token)
1. Download the zip from the competition's **Data** tab on Kaggle (*Download All*) to your computer.
2. Upload it to **Google Drive** (e.g. `MyDrive/`). The zip is ~160 MB, so Drive is faster and survives a Colab restart.
3. Set `ZIP_PATH` below and run. If you would rather upload straight from your computer, set `USE_DRIVE = False`.

In [ ]:
USE_DRIVE = True
ZIP_PATH = '/content/drive/MyDrive/2110531-dsde-2026-1.zip'   # change to where you put the zip

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    from google.colab import files
    uploaded = files.upload()                  # pick the zip in the file dialog
    ZIP_PATH = '/content/' + list(uploaded)[0]

! unzip -q -o "{ZIP_PATH}" -d /content/data
! ls /content/data

# 1. Config

In [ ]:
import os, shutil, random
import numpy as np
import pandas as pd

DATA_DIR  = '/content/data'                 # folder with train.csv, train/, test/, sample_submission.csv
YOLO_DIR  = '/content/yolo_annotations'     # YOLO-format dataset is written here
TRAIN_IMG = os.path.join(DATA_DIR, 'train', 'train')
TEST_IMG  = os.path.join(DATA_DIR, 'test', 'test')

MODEL   = 'yolo11s.pt'
IMGSZ   = 960
EPOCHS  = 50
BATCH   = 16          # T4 16 GB; use 8 if you get CUDA out of memory
CACHE   = 'disk'      # Colab RAM (~12 GB) is too small for cache=True at 960
VAL_CAMS = ['1437', '232']    # held-out cameras = "unseen camera" like the test set
TRAIN_ON_ALL = False          # True for the final model: train on all 15 cameras

IMG_W, IMG_H = 352, 288
NC = 8
class_names = [f'class_{i}' for i in range(NC)]   # 0 = Car, 1 = Motorcycle, ..., 7 = Songthaew
random.seed(42)

# 2. Look at the data

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
print(df.shape)
print(df.class_id.value_counts().sort_index())
df.head()

In [ ]:
all_imgs = sorted(os.listdir(TRAIN_IMG))
cam = lambda f: f.split('_')[0]
print('train images:', len(all_imgs), '| cameras:', sorted({cam(f) for f in all_imgs}))
print('test images :', len(os.listdir(TEST_IMG)), '| cameras:', sorted({cam(f) for f in os.listdir(TEST_IMG)}))
w, h = df.x2 - df.x1, df.y2 - df.y1
print('median box w x h:', w.median(), 'x', h.median())

# 3. Clean the boxes
Clip boxes to the image, drop zero-size boxes and exact duplicates.
Images with no boxes are kept as background examples.

In [ ]:
df[['x1', 'x2']] = df[['x1', 'x2']].clip(0, IMG_W)
df[['y1', 'y2']] = df[['y1', 'y2']].clip(0, IMG_H)
df = df[(df.x2 > df.x1) & (df.y2 > df.y1)]
df = df.drop_duplicates(subset=['image_id', 'class_id', 'x1', 'y1', 'x2', 'y2'])
print('boxes after cleaning:', len(df))

# 4. Convert to YOLO format
Same folder layout as the class notebook:
```
yolo_annotations/
├── images/train, images/val
└── labels/train, labels/val      # one .txt per image: class x_center y_center width height (0-1)
```
Instead of `train_test_split`, we split **by camera** so validation measures performance on unseen cameras.
Rare classes are also repeated (repeat-factor sampling) so the model sees them more often.

In [ ]:
val_files   = [] if TRAIN_ON_ALL else [f for f in all_imgs if cam(f) in VAL_CAMS]
train_files = [f for f in all_imgs if f not in set(val_files)]
print('train:', len(train_files), '| val:', len(val_files))

# repeat-factor sampling: r_class = max(1, sqrt(t / fraction_of_images_with_class)), capped at 4
img_classes = df.groupby('image_id').class_id.apply(set).to_dict()
frac = {c: np.mean([c in img_classes.get(f, ()) for f in train_files]) for c in range(NC)}
r_cls = {c: min(4.0, max(1.0, np.sqrt(0.15 / frac[c]))) if frac[c] > 0 else 1.0 for c in range(NC)}
print('repeat factor per class:', {c: round(r, 2) for c, r in r_cls.items()})

In [ ]:
if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)
for split in ['train', 'val']:
    os.makedirs(os.path.join(YOLO_DIR, 'images', split))
    os.makedirs(os.path.join(YOLO_DIR, 'labels', split))

boxes_by_img = {k: g for k, g in df.groupby('image_id')}

def write_example(image_filename, split, out_name):
    shutil.copy(os.path.join(TRAIN_IMG, image_filename), os.path.join(YOLO_DIR, 'images', split, out_name))
    lines = []
    g = boxes_by_img.get(image_filename)
    if g is not None:
        for r in g.itertuples():
            x_center = (r.x1 + r.x2) / 2.0 / IMG_W
            y_center = (r.y1 + r.y2) / 2.0 / IMG_H
            width    = (r.x2 - r.x1) / IMG_W
            height   = (r.y2 - r.y1) / IMG_H
            lines.append(f'{r.class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}')
    with open(os.path.join(YOLO_DIR, 'labels', split, os.path.splitext(out_name)[0] + '.txt'), 'w') as f:
        f.write('\n'.join(lines))

n = 0
for f in train_files:
    r = max([r_cls[c] for c in img_classes.get(f, ())], default=1.0)
    k = int(r) + (random.random() < r - int(r))
    for j in range(k):
        write_example(f, 'train', f if j == 0 else f'{os.path.splitext(f)[0]}__rep{j}.jpg')
        n += 1
for f in val_files:
    write_example(f, 'val', f)
print('train examples after repeating:', n)

In [ ]:
# Create data.yaml file
yaml_content = f"""
train: {YOLO_DIR}/images/train
val: {YOLO_DIR}/images/{'train' if TRAIN_ON_ALL else 'val'}

nc: {NC}
names: {class_names}
"""
with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content)
print(yaml_content)

# 5. Train YOLO11s at imgsz 960
Same call as the class notebook (`model.train(data='data.yaml', ...)`) with more epochs and a few extra settings:
AdamW + cosine LR, mosaic/mixup augmentation, mosaic switched off for the last 10 epochs.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)  # load a pretrained model (recommended for training)
results = model.train(
    data='/content/data.yaml', imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device='cuda:0',
    optimizer='AdamW', lr0=0.001, cos_lr=True, warmup_epochs=3,
    mosaic=1.0, mixup=0.1, close_mosaic=10, patience=20,
    cache=CACHE, workers=2, seed=42,
    val=not TRAIN_ON_ALL, project='/content/runs/detect', name='y11s_960', exist_ok=True,
)
SAVE_DIR = str(model.trainer.save_dir)
WEIGHTS = os.path.join(SAVE_DIR, 'weights', 'last.pt' if TRAIN_ON_ALL else 'best.pt')
print(WEIGHTS)

In [ ]:
from PIL import Image
Image.open(os.path.join(SAVE_DIR, 'results.png'))

In [ ]:
Image.open(os.path.join(SAVE_DIR, 'confusion_matrix.png'))

In [ ]:
Image.open(os.path.join(SAVE_DIR, 'val_batch0_pred.jpg'))

# 6. Validate with the Kaggle metric
Kaggle scores mAP@0.5 with pycocotools. Predict with a very low `conf` so the full precision–recall curve is kept,
and keep at most 100 boxes per class per image.

In [ ]:
CONF, IOU_NMS, MAX_DET, PER_CLASS = 0.001, 0.6, 300, 100
model = YOLO(WEIGHTS)

def predict(filenames, folder, tta=False):
    out = {}
    for i in range(0, len(filenames), 32):
        chunk = filenames[i:i + 32]
        res = model.predict([os.path.join(folder, f) for f in chunk], imgsz=IMGSZ, conf=CONF, iou=IOU_NMS,
                            max_det=MAX_DET, augment=tta, device='cuda:0', verbose=False)
        for f, r in zip(chunk, res):
            b = r.boxes
            xyxy, conf, cls = b.xyxy.cpu().numpy(), b.conf.cpu().numpy(), b.cls.cpu().numpy().astype(int)
            keep = []
            for c in np.unique(cls):                      # <= 100 boxes per class per image
                idx = np.where(cls == c)[0]
                keep += list(idx[np.argsort(-conf[idx])][:PER_CLASS])
            keep = np.array(keep, dtype=int)
            out[f] = (xyxy[keep], conf[keep], cls[keep])
    return out

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

def coco_map50(preds, files):
    ids = {f: i + 1 for i, f in enumerate(files)}
    gt = {'images': [{'id': ids[f], 'width': IMG_W, 'height': IMG_H} for f in files],
          'categories': [{'id': c + 1} for c in range(NC)], 'annotations': []}
    for k, r in enumerate(df[df.image_id.isin(ids)].itertuples(), 1):
        w, h = r.x2 - r.x1, r.y2 - r.y1
        gt['annotations'].append({'id': k, 'image_id': ids[r.image_id], 'category_id': int(r.class_id) + 1,
                                  'bbox': [r.x1, r.y1, w, h], 'area': w * h, 'iscrowd': 0})
    dt = [{'image_id': ids[f], 'category_id': int(c) + 1, 'score': float(s),
           'bbox': [float(b[0]), float(b[1]), float(b[2] - b[0]), float(b[3] - b[1])]}
          for f, (B, S, C) in preds.items() for b, s, c in zip(B, S, C)]
    cg = COCO(); cg.dataset = gt; cg.createIndex()
    E = COCOeval(cg, cg.loadRes(dt), 'bbox'); E.evaluate(); E.accumulate(); E.summarize()
    p = E.eval['precision'][0, :, :, 0, -1]              # IoU 0.5, all areas, maxDets 100
    print('AP50 per class:', {c: round(float(p[:, c][p[:, c] > -1].mean()), 3) if (p[:, c] > -1).any() else None
                              for c in range(NC)})
    return E.stats[1]

if not TRAIN_ON_ALL:
    val_preds = predict(val_files, TRAIN_IMG)
    print('val mAP50 =', round(coco_map50(val_preds, val_files), 4))

In [ ]:
# Draw predictions on one validation image (same idea as the class notebook)
import cv2
from matplotlib import pyplot as plt
if not TRAIN_ON_ALL:
    f = val_files[0]
    image = cv2.imread(os.path.join(TRAIN_IMG, f))[:, :, ::-1].copy()
    B, S, C = val_preds[f]
    for b, s, c in zip(B, S, C):
        if s < 0.3: continue
        x0, y0, x1, y1 = b.astype(int)
        image = cv2.rectangle(image, (x0, y0), (x1, y1), (20, 255, 12), 1)
        cv2.putText(image, f'{c} {s:.2f}', (x0, y0 - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.3, (20, 255, 12), 1)
    plt.figure(figsize=(10, 8)); plt.imshow(image); plt.axis('off')

# 7. Predict the test set and write `submission.csv`
The `image_id` in `sample_submission.csv` is a long Thai name, while the test files have short names
(`<camera>_<date>_<time>.jpg`). We match them by camera + date + time.

In [ ]:
ss = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
test_files = os.listdir(TEST_IMG)
short_by_key = {(cam(f), '_'.join(f.split('_')[-2:])): f for f in test_files}
long_by_short = {}
for long_name in ss.image_id.unique():
    key = (long_name.split('_')[1], '_'.join(long_name.split('_')[-2:]))
    long_by_short[short_by_key[key]] = long_name
print('matched', len(long_by_short), 'of', ss.image_id.nunique())

In [ ]:
TTA = False   # try True: flips/scales at test time, ~3x slower, often +0.5-1 mAP
test_preds = predict(list(long_by_short), TEST_IMG, tta=TTA)

rows = []
for short_name, long_name in long_by_short.items():
    B, S, C = test_preds[short_name]
    n_before = len(rows)
    for b, s, c in zip(B, S, C):
        x1, y1, x2, y2 = [round(float(v), 2) for v in b]
        if x2 - x1 >= 0.5 and y2 - y1 >= 0.5:
            rows.append([long_name, int(c), round(float(s), 5), x1, y1, x2, y2])
    if len(rows) == n_before:                     # every image must appear at least once
        rows.append([long_name, 0, 0.0001, 0, 0, 1, 1])

sub = pd.DataFrame(rows, columns=['image_id', 'class_id', 'confidence', 'x1', 'y1', 'x2', 'y2'])
sub.insert(0, 'id', range(len(sub)))
assert list(sub.columns) == list(ss.columns)
assert sub.image_id.nunique() == ss.image_id.nunique()
assert sub.groupby(['image_id', 'class_id']).size().max() <= PER_CLASS
sub.to_csv('/content/submission.csv', index=False)
print(sub.shape); sub.head()

In [ ]:
from google.colab import files
files.download('/content/submission.csv')
# then upload submission.csv on the competition page: Submit Prediction

# 8. Next steps
1. Upload `submission.csv` on Kaggle (Submit Prediction) and compare the leaderboard score with the validation mAP50.
2. Final model: set `TRAIN_ON_ALL = True`, re-run from section 1 (no validation; uses `last.pt`).
3. Try `TTA = True` in section 7.